In [7]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

no_preview = False # Toggle to False to use the preview widget

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, \
    PathSolver, RadioMapSolver, subcarrier_frequencies

In [8]:
# Load integrated scene
scene = load_scene(sionna.rt.scene.munich) # Try also sionna.rt.scene.etoile

In [9]:
scene.preview()

In [10]:
scene.objects

{'Heilig_Geist-itu_marble': <sionna.rt.scene_object.SceneObject at 0x780190770940>,
 'Heilig_Geist-itu_metal': <sionna.rt.scene_object.SceneObject at 0x780190770880>,
 'Frauenkirche-itu_marble': <sionna.rt.scene_object.SceneObject at 0x780190771210>,
 'Frauenkirche-itu_metal': <sionna.rt.scene_object.SceneObject at 0x7801907712d0>,
 'St__Peter-itu_marble': <sionna.rt.scene_object.SceneObject at 0x780190771480>,
 'St__Peter-itu_metal': <sionna.rt.scene_object.SceneObject at 0x7801907713c0>,
 'ground': <sionna.rt.scene_object.SceneObject at 0x7801907700a0>,
 'no-name-9': <sionna.rt.scene_object.SceneObject at 0x7801907716c0>,
 'no-name-10': <sionna.rt.scene_object.SceneObject at 0x780190771780>,
 'no-name-11': <sionna.rt.scene_object.SceneObject at 0x780190771840>,
 'no-name-12': <sionna.rt.scene_object.SceneObject at 0x780190771900>}

In [11]:
building = scene.get("Heilig_Geist-itu_marble")

In [13]:
building.radio_material

ITURadioMaterial type=marble
                 eta_r=7.074
                 sigma=0.018
                 thickness=0.100
                 scattering_coefficient=0.000
                 xpd_coefficient=0.000

In [14]:
scene = load_scene(sionna.rt.scene.munich, merge_shapes=True) # Merge shapes to speed-up computations

# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="cross")

# Create transmitter
tx = Transmitter(name="tx",
                 position=[8.5,21,27],
                 display_radius=2)

# Add transmitter instance to scene
scene.add(tx)

# Create a receiver
rx = Receiver(name="rx",
              position=[45,90,1.5],
              display_radius=2)

# Add receiver instance to scene
scene.add(rx)

tx.look_at(rx) # Transmitter points towards receiver

In [15]:
# Instantiate a path solver
# The same path solver can be used with multiple scenes
p_solver  = PathSolver()

# Compute propagation paths
paths = p_solver(scene=scene,
                 max_depth=5,
                 los=True,
                 specular_reflection=True,
                 diffuse_reflection=False,
                 refraction=True,
                 synthetic_array=False,
                 seed=41)

In [16]:
if no_preview:
    scene.render(camera=my_cam, paths=paths, clip_at=20);
else:
    scene.preview(paths=paths, clip_at=20);

In [17]:
rm_solver = RadioMapSolver()

rm = rm_solver(scene=scene,
               max_depth=5,
               cell_size=[1,1],
               samples_per_tx=10**6)

In [18]:
if no_preview:
    scene.render(camera=my_cam, radio_map=rm);
else:
    scene.preview(radio_map=rm);